# Day 1 - Lesson 6 실습: 응답 스트리밍

> **이론 문서**: `Day1_Lesson6_Theory.md`  
> **선행 실습**: `Day1_Lesson1` ~ `Day1_Lesson5` 완료  
> **실습 시간**: 약 70분  
> **사용 모델**: GPT-4o-mini

---

## 🎯 이 노트북에서 만들 것

5교시 챗봇은 답변이 **5초 후 한 번에** 나타났어요.  
오늘은 ChatGPT처럼 **한 글자씩 흘러나오게** 만듭니다.

오늘은 **천천히, 하나씩** 익혀갑니다. 새로운 개념은 한 단계에 하나씩만 추가해요.

```
1️⃣ invoke vs stream — 눈으로 차이 보기
       ↓
2️⃣ chunk 누적 패턴 — 토큰을 모아서 전체 답변 만들기
       ↓
3️⃣ ⭐ yield 체험 — LLM 없이 Gradio가 어떻게 화면을 갱신하는지 먼저 이해
       ↓
4️⃣ ChatInterface 스트리밍 — 가장 쉬운 패턴
       ↓
5️⃣ 메모리 + 스트리밍 — 4교시 코드를 stream으로만 바꾸기
       ↓
6️⃣ Blocks 스트리밍 (단순) — Textbox에 답변 점진 출력
       ↓
7️⃣ Blocks 스트리밍 (Chatbot) — 채팅 UI에 적용
       ↓
8️⃣ 토큰 사용량 확인
       ↓
🎁 도전과제: 페르소나 + 메모리 + 스트리밍 챗봇
```

## 📖 이론 매핑

| 이론 문서 섹션 | 노트북 |
|---|---|
| 1부 + 2부. 개념 비교 | Step 1 |
| 3부. chunk 구조 | Step 2 |
| 4부. yield 패턴 | Step 3·4 |
| 4부. 메모리 + 스트리밍 | Step 5 |
| 4부. Blocks 스트리밍 | Step 6·7 |
| Q4. usage_metadata | Step 8 |
| 5부. 결과물 명세 | 도전과제 |

---

## 💡 오늘 외울 한 줄

> ```python
> full = ""
> for chunk in chain.stream(...):
>     full += chunk.content
>     yield full   # ← return이 아닌 yield, 그리고 누적된 전체를 보냄
> ```


---

# 🛠 Step 0. 환경 준비


In [2]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


In [1]:
# 2️⃣ 패키지 설치
!pip install -q gradio langchain langchain-openai langchain-core python-dotenv



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 3️⃣ API 키 로드
import os
if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


✅ API 키 OK


In [4]:
# 4️⃣ 공통 도구 준비
from langchain_openai import ChatOpenAI
import gradio as gr
import time

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print(f"✅ Gradio {gr.__version__} · LLM 준비 완료")


c:\workspace\chatbot\langchain-web-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Gradio 6.29.0 · LLM 준비 완료


---

# 1️⃣ Step 1. `invoke` vs `stream` — 눈으로 차이 보기

> **이론 매핑**: 1부 + 2부

같은 질문을 두 가지 방식으로 호출해봅시다. **체감 차이**가 핵심이에요.

### 🅰️ `invoke` — 다 만들어질 때까지 기다리기


In [5]:
# invoke — 5초 정도 기다린 후 한 번에 답변
question = "Python을 처음 배우는 사람에게 추천하는 학습 순서를 알려주세요."

print("⏳ 호출 시작...")
start = time.time()
result = llm.invoke(question)
print(f"\n[{time.time() - start:.1f}초 후 한 번에 도착]")
print(result.content)


⏳ 호출 시작...

[8.7초 후 한 번에 도착]
Python을 처음 배우는 사람에게 추천하는 학습 순서는 다음과 같습니다:

### 1. Python 설치 및 환경 설정
- **Python 설치**: Python 공식 웹사이트에서 최신 버전을 다운로드하고 설치합니다.
- **IDE 선택**: PyCharm, VSCode, Jupyter Notebook 등 편리한 개발 환경을 선택합니다.

### 2. 기본 문법 이해
- **변수와 데이터 타입**: 정수, 실수, 문자열, 리스트, 튜플, 딕셔너리 등 기본 데이터 타입을 배우기.
- **연산자**: 산술 연산자, 비교 연산자, 논리 연산자 등을 익히기.
- **조건문**: `if`, `elif`, `else` 문을 사용하여 조건에 따른 분기 처리 배우기.
- **반복문**: `for` 루프와 `while` 루프를 사용하여 반복 작업 수행하기.

### 3. 함수와 모듈
- **함수 정의**: `def` 키워드를 사용하여 함수를 정의하고 호출하는 방법 배우기.
- **모듈 사용**: Python 내장 모듈과 외부 모듈을 임포트하여 사용하는 방법 익히기.

### 4. 데이터 구조
- **리스트, 튜플, 딕셔너리, 집합**: 각 데이터 구조의 특징과 사용법을 익히기.
- **리스트 컴프리헨션**: 리스트를 간결하게 생성하는 방법 배우기.

### 5. 파일 입출력
- **파일 읽기/쓰기**: 텍스트 파일을 읽고 쓰는 방법 배우기.

### 6. 예외 처리
- **try-except 블록**: 오류가 발생할 수 있는 코드에서 예외를 처리하는 방법 배우기.

### 7. 객체 지향 프로그래밍 (OOP)
- **클래스와 객체**: 클래스 정의, 객체 생성, 메서드 사용법 배우기.
- **상속과 다형성**: 상속을 통해 코드 재사용하는 방법 배우기.

### 8. 라이브러리 활용
- **NumPy**: 수치 계산을 위한 라이브러리 사용법 배우기.
- **Pandas**: 데이터 분석을 위한 라이브러리 사용법 배우기.
-

✅ **결과 확인**: 빈 화면 → 한 번에 긴 답변. "응답이 오긴 하나?" 하는 답답함이 있었죠?

### 🎲 예측 게임 — `stream`은 어떻게 다를까?

다음 셀에서 *같은 질문*을 `stream`으로 호출합니다. 실행 *전에* 예측해보세요:

| 특성 | invoke (위에서 본 것) | stream (다음 셀) |
|---|:---:|:---:|
| 첫 토큰 도착 시간 | ~5초 (한 번에 옴) | __초 |
| **전체 완료** 시간 | ~5초 | ☐ 더 빠름 / ☐ 같음 / ☐ 더 느림 |
| **사용자 체감** 속도 | 느림 | ☐ 더 빠름 / ☐ 같음 / ☐ 더 느림 |
| 답변 내용 (길이/품질) | 현재 본 그대로 | ☐ 더 김 / ☐ 같음 / ☐ 더 짧음 |

> 💡 *전체 완료 시간*과 *체감 속도*가 **다른 결과**로 나올 수 있다는 점에 주목하세요. 스트리밍의 핵심이에요.

### 🅱️ `stream` — 토큰이 도착하는 즉시 출력

In [6]:
# stream — for 루프로 chunk를 하나씩 받아 즉시 출력
print("⏳ 호출 시작...")
start = time.time()
first = None

for chunk in llm.stream(question):
    if first is None:
        first = time.time() - start
        print(f"[첫 토큰: {first:.2f}초]\n")
    print(chunk.content, end="", flush=True)  # end="" → 줄바꿈 X, flush=True → 즉시 출력

print(f"\n\n[전체 완료: {time.time() - start:.1f}초]")


⏳ 호출 시작...
[첫 토큰: 0.43초]

Python을 처음 배우는 사람에게 추천하는 학습 순서는 다음과 같습니다:

1. **기본 개념 이해하기**
   - 프로그래밍의 기본 개념 (변수, 데이터 타입, 조건문, 반복문 등)
   - Python의 설치 및 환경 설정 (Anaconda, Jupyter Notebook, VSCode 등)

2. **기본 문법 익히기**
   - 변수와 데이터 타입 (정수, 실수, 문자열, 리스트, 튜플, 딕셔너리, 집합)
   - 연산자 (산술, 비교, 논리 연산자)
   - 조건문 (if, elif, else)
   - 반복문 (for, while)

3. **함수와 모듈**
   - 함수 정의 및 호출
   - 매개변수와 반환값
   - 내장 함수 및 표준 라이브러리 사용법
   - 모듈과 패키지 개념

4. **객체지향 프로그래밍 (OOP)**
   - 클래스와 객체의 개념
   - 속성과 메서드
   - 상속과 다형성

5. **파일 입출력**
   - 텍스트 파일 읽기 및 쓰기
   - CSV 파일 처리

6. **예외 처리**
   - try, except 문을 사용한 예외 처리
   - 사용자 정의 예외

7. **라이브러리 및 프레임워크 활용**
   - 데이터 분석: NumPy, Pandas
   - 데이터 시각화: Matplotlib, Seaborn
   - 웹 개발: Flask, Django (기본 개념)

8. **프로젝트 진행하기**
   - 간단한 프로젝트를 통해 배운 내용을 적용해보기 (예: 계산기, 웹 스크래퍼, 데이터 분석 프로젝트 등)
   - Git과 GitHub를 사용하여 버전 관리 및 협업 경험 쌓기

9. **심화 학습**
   - 알고리즘 및 자료구조
   - 웹 API 사용법
   - 데이터베이스 연동 (SQLite, MySQL 등)

10. **커뮤니티 참여 및 지속적인 학습**
    - Stack Overflow, GitHub, Reddit 등에서 질문하고 답변하기
    

✅ **결과 확인**: 첫 토큰이 **0.5~1초 안에** 도착했나요?

> 💡 핵심: 스트리밍은 **빠른 게 아니라 빨라 보이는** 기술입니다.  
> 같은 5초인데도 "이미 받고 있다"는 느낌이 들죠.


---

# 2️⃣ Step 2. `chunk` 누적 패턴

> **이론 매핑**: 3부. chunk 구조 + 누적

스트리밍에서 받는 `chunk`는 **1~5글자짜리 작은 조각**이에요.  
이걸 **이어 붙여서 전체 답변**을 만드는 게 핵심 패턴입니다.


### 🤔 실행 전에 1분 멈춤

다음 셀에서 `chunk`라는 변수의 정체를 들여다봅니다.  
실행하기 *전에* 예측해보세요:

- `chunk`의 *타입*은? **그냥 문자열? 딕셔너리? 다른 객체?**
- 위의 누적 패턴 `full += chunk.content`에서 `.content`를 붙인 *이유*는?
- 첫 chunk의 `content`는 어떤 모양일까? **글자? 빈 문자열? 다른 것?**

예측을 마음에 두고 실행하세요.

In [7]:
# 작은 조각이 어떻게 생겼는지 처음 5개만 확인
for i, chunk in enumerate(llm.stream("AI를 한 문장으로 설명해주세요.")):
    if i == 0:
        print(f"📦 chunk의 타입: {type(chunk).__name__}")
        print()
    print(f"chunk {i}: content={chunk.content!r}")
    if i >= 4:
        break

📦 chunk의 타입: AIMessageChunk

chunk 0: content=''
chunk 1: content='AI'
chunk 2: content='('
chunk 3: content='인'
chunk 4: content='공지'


각 chunk는 보통 1~5글자입니다. 이걸 **이어 붙이면** 전체 답변이 돼요.


In [8]:
# 누적 패턴 — 화면 출력 + 변수 누적을 동시에
full = ""
for chunk in llm.stream("LangChain을 한 문장으로 설명해주세요."):
    full += chunk.content                    # 변수에 누적
    print(chunk.content, end="", flush=True) # 화면에 즉시 출력

print(f"\n\n📦 누적된 전체 길이: {len(full)}자")


LangChain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축하고 연결할 수 있도록 돕는 프레임워크입니다.

📦 누적된 전체 길이: 71자


✅ **결과 확인**: 화면엔 글자가 흘러나오면서, `full` 변수엔 전체 답변이 모였습니다.

> 💡 이 **누적 패턴**이 곧 Gradio 스트리밍의 정체예요.


---

# 3️⃣ Step 3. ⭐ `yield` 체험 — LLM 없이 먼저 이해하기

> **이론 매핑**: 4부. Gradio yield 패턴

Gradio 스트리밍의 핵심은 **`yield`** 입니다. 그런데 yield는 LLM이랑 상관없는 **Python 문법**이에요.  
LLM을 끼우기 전에 **yield 그 자체**를 한 번 체험해봅시다.

### yield란?

```python
return  → 함수가 끝남, 결과 1개를 한 번에 반환
yield   → 함수가 안 끝남, 결과를 여러 번 나눠서 반환
```

### 🧪 실험: 1초마다 숫자를 yield 하는 함수

LLM 없이 그냥 **숫자를 천천히 세는** 함수를 만들어 Gradio가 어떻게 화면을 갱신하는지 봅시다.


In [9]:
# yield 본질 체험 — 1부터 5까지 천천히 세기 (LLM 아님!)
def slow_count(target):
    """target까지 1초마다 숫자를 yield"""
    text = ""
    for i in range(1, int(target) + 1):
        text += f"{i} "       # 누적
        time.sleep(0.5)        # 0.5초 대기 (LLM 토큰 생성 시간 시뮬레이션)
        yield text             # 누적된 텍스트 yield


gr.close_all()

demo = gr.Interface(
    fn=slow_count,
    inputs=gr.Number(value=5, label="몇까지 셀까요?"),
    outputs=gr.Textbox(label="출력"),
    title="🌊 yield 체험 (LLM 없음)",
    description="숫자가 0.5초마다 하나씩 추가되는 걸 확인하세요"
)
demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


✅ **결과 확인**: 숫자가 **하나씩 천천히 추가**되며 화면이 갱신되나요?

### 🔑 여기서 깨달아야 할 3가지

1. **`yield`를 만나면 Gradio가 화면을 즉시 업데이트**합니다
2. **누적된 전체**를 yield 해야 합니다 (`text += ...` 후 `yield text`)
3. LLM 스트리밍도 이 패턴과 똑같아요 — `time.sleep(0.5)` 자리에 **`for chunk in llm.stream(...)`** 이 들어갈 뿐


---

# 4️⃣ Step 4. ChatInterface 스트리밍 — 가장 단순한 패턴

> **이론 매핑**: 4부

이제 Step 3의 `yield` 패턴에 LLM을 끼웁니다.  
`time.sleep(0.5)` 자리에 **`for chunk in llm.stream(...)`** 만 넣으면 끝.


In [10]:
# 가장 단순한 스트리밍 챗봇 (메모리 없음)
def streaming_chat(message, history):
    """return 대신 yield, 누적해서 보내기"""
    full = ""
    for chunk in llm.stream(message):
        full += chunk.content
        yield full   # ← Step 3의 yield와 똑같은 패턴!


gr.close_all()

demo = gr.ChatInterface(
    fn=streaming_chat,
    title="🌊 스트리밍 챗봇",
    description="답변이 ChatGPT처럼 한 글자씩 나타나는지 확인",
)
demo.launch()


Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


✅ **결과 확인**: 답변이 **한 글자씩 흘러나오나요?**

> 💡 ChatInterface는 yield를 받을 때마다 자동으로 화면 갱신.  
> 우리가 한 일은 5교시 코드에서 **`return result.content` → `yield full`** 로 바꾼 게 전부예요.


---

### 🐛 잠깐, 이상한 코드 하나 — *조용한* 버그 (스트리밍 버전)

방금 본 정상 패턴(`yield full`)에서 *한 단어만* 다른 버전입니다.  
에러는 안 나지만 챗봇과 대화해보면 *답변이 깜빡거리거나 마지막 몇 글자만* 보일 거예요.

**AI에게 물어보기 전에**, 코드를 5분만 노려보세요. 어디가 문제일까요?

> 💡 Step 3의 `slow_count` 깨달음 #2를 다시 떠올려보세요 — "**누적된 전체**를 yield 해야 합니다".

In [11]:
# 🐛 스트리밍 챗봇 — 그런데 답변이 깜빡임
def buggy_streaming_chat(message, history):
    full = ""
    for chunk in llm.stream(message):
        full += chunk.content
        yield full   # ← 한 단어 다름. 어디가 문제?


gr.close_all()

demo = gr.ChatInterface(
    fn=buggy_streaming_chat,
    title="🐛 버그 챗봇 — 어디가 잘못됐을까?",
    description="메시지를 보내보세요. 답변이 어떻게 나오나요?",
)
demo.launch()

Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


🤔 **관찰 포인트**:
- 챗봇에 "AI가 뭐예요?" 같은 질문을 보내보세요. 답변이 어떻게 나오나요?
- *깜빡이거나* *마지막 1-5글자만* 보이지 않나요?
- `yield chunk.content`는 매 토큰마다 *그 토큰만* 화면에 보냅니다. Gradio는 새 yield가 올 때마다 *기존 표시를 덮어쓰기* 때문에…

> 💡 **교훈**: Gradio 스트리밍은 *지금까지의 누적 상태*를 매번 yield 해야 합니다.  
> `yield chunk.content` (1-5자만) → `yield full` (누적된 전체)로 바꾸면 해결.
>
> **버그 시리즈 진행 상황**:
> - 1교시 분류기: 조용한 버그 (temperature)
> - 2교시 템플릿: 시끄러운 `KeyError` (변수 이름)
> - 4교시 메모리: 시끄러운 `ValueError` (config 누락)
> - 5교시 Gradio: 조용한 버그 (inputs 순서)
> - **6교시 스트리밍: 조용한 버그 (yield 대상이 잘못됨)** ← 오늘
>
> 스트리밍 버그는 *시각적으로 비정상*이라 빨리 발견됩니다 — 사용자가 못 알아채는 종류는 아니에요.

#### ✏️ 직접 고쳐보기

위 셀을 *복사해서* 아래에 새 셀을 만들고 `yield`만 바로잡아 동작하게 고쳐보세요.

---

# 5️⃣ Step 5. 메모리 + 스트리밍 — 4교시 코드를 그대로!

> **이론 매핑**: 4부. 메모리 챗봇 + 스트리밍

4교시 메모리 체인에 `invoke` → `stream`만 바꾸면 끝.  
**다른 코드는 한 글자도 안 바뀝니다.**


In [12]:
# 4교시 메모리 체인 재구성 (기억 안 나면 Day1_Lesson4 참고)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

store = {}
def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    memory_prompt | llm,
    get_history,
    input_messages_key="question",
    history_messages_key="history",
)


c:\workspace\chatbot\langchain-web-chatbot\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
# 메모리 + 스트리밍 — Step 4와 거의 같음 (llm.stream → chain_with_memory.stream)
def streaming_memory_chat(message, history):
    full = ""
    for chunk in chain_with_memory.stream(
        {"question": message},
        config={"configurable": {"session_id": "default"}},
    ):
        full += chunk.content
        yield full


gr.close_all()

demo = gr.ChatInterface(
    fn=streaming_memory_chat,
    title="🤖 메모리 + 스트리밍",
    description="대화를 기억하면서 답변은 점진적으로 표시",
)
demo.launch()


Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### 🧪 테스트 시나리오

1. `안녕! 내 이름은 진환이야`
2. `취미는 모델 비행기 만들기야`
3. `내 이름이랑 취미가 뭐였지?` ← **둘 다 기억 + 스트리밍**

✅ **결과 확인**: 답변이 글자 단위로 흘러나오면서도 이전 대화를 정확히 기억하나요?


---

# 6️⃣ Step 6. Blocks 스트리밍 (단순 버전) — Textbox 출력

> **이론 매핑**: 4부. Blocks에서 스트리밍

ChatInterface는 yield 한 줄로 끝이었어요.  
**Blocks도 똑같이 단순할 수 있습니다** — 출력이 그냥 `Textbox`라면.

### 핵심: Step 3 `slow_count`와 코드 구조가 똑같습니다


In [14]:
# Blocks + Textbox 스트리밍 — Step 3 slow_count 패턴 그대로
def stream_to_textbox(question):
    """질문을 받아 Textbox에 답변을 점진 출력"""
    full = ""
    for chunk in llm.stream(question):
        full += chunk.content
        yield full


gr.close_all()

with gr.Blocks(title="Blocks 스트리밍 단순") as demo:
    gr.Markdown("# 🪟 Blocks 스트리밍 (단순 버전)")
    gr.Markdown("질문을 입력하면 아래 텍스트박스에 답변이 점진적으로 표시됩니다.")

    inp = gr.Textbox(label="질문", placeholder="여기에 입력...")
    btn = gr.Button("전송", variant="primary")
    out = gr.Textbox(label="답변", lines=10, interactive=False)

    btn.click(stream_to_textbox, inputs=inp, outputs=out)
    inp.submit(stream_to_textbox, inputs=inp, outputs=out)

demo.launch()


Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


✅ **결과 확인**: Textbox에 답변이 점진적으로 채워졌나요?

> 💡 함수 본체는 **Step 4 ChatInterface 버전과 똑같습니다.**  
> 차이는 호출 쪽: ChatInterface는 알아서 채팅 UI에 표시, Blocks는 우리가 outputs로 어디에 표시할지 지정.


---

# 7️⃣ Step 7. Blocks + `gr.Chatbot` 스트리밍

이제 Textbox 대신 **채팅 UI(`gr.Chatbot`)**에 스트리밍 답변을 표시해봅시다.  
하나만 추가되는 개념: **history 리스트에 메시지를 직접 넣고 마지막 자리를 업데이트**.

### `gr.Chatbot`이 기대하는 history 형식

```python
[
    {"role": "user", "content": "안녕"},          # 사용자 메시지
    {"role": "assistant", "content": "안녕하세요!"}, # AI 답변
    ...
]
```

### 새 답변을 추가하는 과정

```
1. 사용자 메시지 추가:   history + [{user, "안녕"}]
2. AI 빈자리 추가:       history + [{user, "안녕"}, {assistant, ""}]
3. 스트리밍하며 빈자리 채우기: history[-1]["content"] = full
4. 매번 yield history
```


In [15]:
# Blocks + Chatbot 스트리밍 — history 직접 업데이트
def stream_to_chatbot(message, history):
    """채팅창에 사용자 메시지 + AI 답변(점진)을 추가"""
    # 1) 사용자 메시지 + AI 빈자리 추가
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ""},
    ]

    # 2) 스트리밍하며 마지막(AI) 메시지의 content만 갱신
    full = ""
    for chunk in llm.stream(message):
        full += chunk.content
        history[-1]["content"] = full
        yield history


In [17]:
gr.close_all()

with gr.Blocks(title="Chatbot 스트리밍") as demo:
    gr.Markdown("# 💬 Blocks + Chatbot 스트리밍")

    chatbot = gr.Chatbot(label="대화창", height=400)
    msg_box = gr.Textbox(label="메시지", placeholder="입력 후 엔터")
    btn = gr.Button("전송", variant="primary")

    btn.click(stream_to_chatbot, [msg_box, chatbot], chatbot)
    msg_box.submit(stream_to_chatbot, [msg_box, chatbot], chatbot)

demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


✅ **결과 확인**: 사용자 메시지가 추가된 후, AI 답변이 그 아래에 **글자 단위로 흘러나오나요?**

> 💡 두 단계만 새로 추가됐어요:
> 1. `history + [...]`로 사용자/AI 자리 추가
> 2. `history[-1]["content"] = full`로 마지막 자리만 갱신

> ⚠️ **메모리는 아직 없습니다** (그냥 `llm.stream` 사용). 도전과제에서 메모리를 결합합니다.


---

# 8️⃣ Step 8. 토큰 사용량 확인

> **이론 매핑**: Q4. usage_metadata

답변마다 토큰을 얼마나 썼는지 확인하는 법입니다.  
**가장 단순한 방법은 `invoke`로 호출**하는 거예요. 그 결과에 `usage_metadata`가 들어있습니다.


In [18]:
# invoke로 호출 → 토큰 정보 바로 확인
result = llm.invoke("AI의 역사를 3문장으로 설명해주세요.")

print(result.content)
print()
print(f"📊 토큰: {result.usage_metadata}")


AI의 역사는 1950년대에 앨런 튜링의 "튜링 테스트" 개념과 함께 시작되었으며, 초기에는 기초적인 규칙 기반 시스템과 문제 해결 알고리즘이 개발되었습니다. 1980년대와 1990년대에는 신경망과 기계 학습의 발전으로 AI 연구가 다시 활성화되었고, 데이터와 컴퓨팅 파워의 증가로 인해 2010년대에는 딥러닝 기술이 혁신적인 성과를 이루었습니다. 현재 AI는 자연어 처리, 이미지 인식, 자율주행차 등 다양한 분야에서 활용되며, 인간의 삶을 변화시키고 있습니다.

📊 토큰: {'input_tokens': 19, 'output_tokens': 148, 'total_tokens': 167, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### 💰 원화 비용 계산


In [33]:
# GPT-4o-mini 가격표 (1M 토큰당 USD)
PRICE_IN = 0.15
PRICE_OUT = 0.60
USD_KRW = 1400

u = result.usage_metadata
cost_krw = (u["input_tokens"]/1_000_000*PRICE_IN + u["output_tokens"]/1_000_000*PRICE_OUT) * USD_KRW
print(f"💰 이번 호출 비용: 약 {cost_krw:.4f}원")


💰 이번 호출 비용: 약 0.1048원


✅ **결과 확인**: 호출 1회 비용이 1원 미만이죠?

> 💡 스트리밍 중에도 토큰을 받을 수 있지만, 구현이 조금 복잡해요.  
> **간단하게 가려면 invoke로 호출하고 토큰을 보여주는** 방식이 권장됩니다.  
> 챗봇에서는 답변 표시는 스트리밍, 토큰 표시는 마지막 한 번만 — 이런 식으로 분리해도 충분합니다.


---

# 📝 정리

### 외울 패턴 2개

```python
# 패턴 A: ChatInterface (가장 단순)
def chat_fn(message, history):
    full = ""
    for chunk in chain.stream({"question": message}):
        full += chunk.content
        yield full

# 패턴 B: Blocks + Chatbot
def respond(message, history):
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ""},
    ]
    full = ""
    for chunk in chain.stream({"question": message}):
        full += chunk.content
        history[-1]["content"] = full
        yield history
```

| 만들 것 | 추천 패턴 |
|---|---|
| 챗봇 화면만 필요 | 패턴 A (ChatInterface) |
| 페르소나 드롭다운 + 채팅창 같은 복합 UI | 패턴 B (Blocks + Chatbot) |

7교시 결과물은 페르소나 드롭다운이 있으니 **패턴 B**를 씁니다.

### 다음 시간: 7교시 자율 실습

7교시는 **본인이 직접 만드는** 시간이에요. 도전과제가 그대로 출발점이 됩니다.

---

도전과제로 손에 익혀봅시다 🎯


---

# 🎁 도전과제 1. 페르소나 + 메모리 + 스트리밍 (ChatInterface 버전)

먼저 **가장 쉬운 ChatInterface 버전**으로 만들어봅시다.  
페르소나 드롭다운은 잠시 잊고, **메모리 + 스트리밍 + 페르소나(고정)** 부터 익혀요.

### 힌트
- Step 5의 메모리 체인 그대로 사용
- system 메시지에 페르소나를 박아둠 (드롭다운 없이 한 종류)
- `stream`으로 토큰 받기 → `yield full`


In [19]:
# 📌 도전 1 — 빈칸을 채우세요
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 친한 친구 페르소나
friend_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 사용자의 절친한 친구입니다. 반말로 편하게 답하세요."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

friend_store = {}
def get_friend_history(session_id):
    if session_id not in friend_store:
        friend_store[session_id] = InMemoryChatMessageHistory()
    return friend_store[session_id]

friend_chain = RunnableWithMessageHistory(
    friend_prompt | llm,
    get_friend_history,
    input_messages_key="question",
    history_messages_key="history",
)


def friend_chat(message, history):
    """친구 페르소나 + 메모리 + 스트리밍"""
    full = ""
    # 👇 빈칸: friend_chain을 stream으로 호출 (session_id="default")
    #for chunk in "___________________":  # ← 빈칸
    for chunk in friend_chain.stream(
        {"question": message},
        config={"configurable": {"session_id": "default"}}):   # ← 빈칸
        full += chunk.content
        yield full


gr.close_all()

demo = gr.ChatInterface(
    fn=friend_chat,
    title="🧑‍🤝‍🧑 친구 챗봇",
    description="반말로 대화하면서 기억까지 하는 친구",
)
demo.launch()


c:\workspace\chatbot\langchain-web-chatbot\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### 🧪 테스트
1. `안녕! 내 이름은 진환이야`
2. `요즘 데이터 분석 공부 중이야`
3. `내가 누군지, 뭐 하는지 기억해?` → **둘 다 기억 + 반말 + 스트리밍**

✅ **결과 확인**: 세 가지가 모두 동작하나요?


---

# 🎁 도전과제 2. 페르소나 챗봇 — 3가지 스트리밍 전략 비교 ⭐

> **이론 매핑**: 4부. Blocks 스트리밍 + 5부. 결과물 명세

5교시 페르소나 챗봇에 6교시 스트리밍을 결합합니다.  
이번엔 **3가지 스트리밍 전략**을 모두 만들어서 *어느 게 가장 UX가 좋은가* 판단해보세요.

| 전략 | yield 시점 | 예상 UX |
|---|---|---|
| 🅰️ **즉시 스트리밍** | chunk마다 yield | 글자가 한 개씩 흐름 (ChatGPT 같음) |
| 🅱️ **배치 스트리밍** | chunk 10개 모아서 yield | 단어 덩어리로 끊김 |
| 🆑 **일괄 표시** | 다 받은 뒤 마지막에 한 번만 yield | invoke와 똑같이 느껴짐 |

같은 챗봇에서 라디오 버튼으로 전략만 바꿔서 *같은 질문*을 던져보고 체감을 비교하세요.

In [22]:
# 📌 페르소나 사전 + 체인 설정
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

PERSONAS = {
    "꼼꼼한 분석가": "당신은 시니어 데이터 분석가입니다. 수치와 근거 중심으로 답하세요.",
    "친절한 선생님": "당신은 초등학생에게 설명하는 선생님입니다. 쉬운 비유로 답하세요.",
    "친한 친구": "당신은 절친한 친구입니다. 반말로 편하게 답하세요.",
    # 🎯 미션: 본인만의 4번째 페르소나를 추가하세요.
    #         (예시: "꼬치꼬치 캐묻는 시니어 리뷰어", "츤데레 데이터 멘토" 등)
    #         조건: system 메시지를 *2문장 이상*으로 구체화할 것
    # "____": "____",
    "츤데레 개발 선배": "당신은 겉으로는 무뚝뚝하지만 속으로는 후배의 성장을 진심으로 응원하는 10년 차 시니어 개발자입니다. 질문에 답할 때는 먼저 문제점을 날카롭게 지적하고, 마지막에는 항상 실용적인 해결책과 함께 따뜻한 조언을 건네주세요.",
}

persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "{persona_desc}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

persona_store = {}
def get_persona_history(session_id):
    if session_id not in persona_store:
        persona_store[session_id] = InMemoryChatMessageHistory()
    return persona_store[session_id]

persona_chain = RunnableWithMessageHistory(
    persona_prompt | llm,
    get_persona_history,
    input_messages_key="question",
    history_messages_key="history",
)

c:\workspace\chatbot\langchain-web-chatbot\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [20]:
# 🎯 3가지 스트리밍 전략 — 셋 다 정의

def respond_immediate(message, history, persona):
    """🅰️ 즉시 스트리밍 — chunk마다 yield (정석 패턴)"""
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ""},
    ]
    full = ""
    for chunk in persona_chain.stream(
        {"question": message, "persona_desc": PERSONAS[persona]},
        config={"configurable": {"session_id": f"persona_{persona}"}},
    ):
        full += chunk.content
        history[-1]["content"] = full
        yield history  # ← 매 chunk마다


def respond_batched(message, history, persona):
    """🅱️ 배치 스트리밍 — chunk 10개 모아서 yield"""
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ""},
    ]
    full = ""
    count = 0
    for chunk in persona_chain.stream(
        {"question": message, "persona_desc": PERSONAS[persona]},
        config={"configurable": {"session_id": f"persona_{persona}"}},
    ):
        full += chunk.content
        count += 1
        if count >= 10:  # ← 10개마다만 yield
            history[-1]["content"] = full
            yield history
            count = 0
    # 잔여 chunk 마지막에 한 번 더
    history[-1]["content"] = full
    yield history


def respond_bulk(message, history, persona):
    """🆑 일괄 표시 — 다 받은 뒤 마지막에 한 번만 yield"""
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ""},
    ]
    full = ""
    for chunk in persona_chain.stream(
        {"question": message, "persona_desc": PERSONAS[persona]},
        config={"configurable": {"session_id": f"persona_{persona}"}},
    ):
        full += chunk.content
        # ← 루프 안에서 yield 안 함
    # 끝나고 단 한 번만 yield (invoke와 동일한 체감)
    history[-1]["content"] = full
    yield history

In [24]:
# UI — 라디오 버튼으로 전략을 골라가며 비교
def respond_with_strategy(message, history, persona, strat):
    """선택된 전략으로 분기"""
    if strat == "A":
        yield from respond_immediate(message, history, persona)
    elif strat == "B":
        yield from respond_batched(message, history, persona)
    else:  # "C"
        yield from respond_bulk(message, history, persona)


gr.close_all()

with gr.Blocks(title="3가지 스트리밍 전략 비교") as demo:
    gr.Markdown("# 🎭 페르소나 챗봇 — 3가지 스트리밍 전략")
    gr.Markdown(
        "같은 질문을 *전략 A → B → C* 순서로 보내며 체감 차이를 확인하세요.  \n"
        "(같은 페르소나 안에서는 메모리가 공유됩니다)"
    )

    with gr.Row():
        persona_dd = gr.Dropdown(
            choices=list(PERSONAS.keys()),
            value="친한 친구",
            label="페르소나",
        )
        strategy = gr.Radio(
            choices=[
                ("🅰️ 즉시 스트리밍 (chunk마다)", "A"),
                ("🅱️ 배치 스트리밍 (10 chunks마다)", "B"),
                ("🆑 일괄 표시 (마지막에 한 번만)", "C"),
            ],
            value="A",
            label="스트리밍 전략",
        )

    chatbot = gr.Chatbot(label="대화창", height=400)
    msg_box = gr.Textbox(label="메시지", placeholder="입력 후 엔터")
    btn = gr.Button("전송", variant="primary")

    btn.click(
        respond_with_strategy,
        [msg_box, chatbot, persona_dd, strategy],
        chatbot,
    )
    msg_box.submit(
        respond_with_strategy,
        [msg_box, chatbot, persona_dd, strategy],
        chatbot,
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### 🧪 판단 시나리오

다음 절차를 *세 가지 전략 각각*에 대해 반복해보세요 (같은 페르소나 유지):

1. 전략을 🅰️로 선택
2. "데이터 분석을 5문장으로 설명해줘" 같은 *답변이 좀 긴* 질문 던지기
3. *답변이 화면에 어떻게 나타나는지* 관찰 (글자 한 개씩? 덩어리로? 한 번에?)
4. 전략을 🅱️로 바꾸고 같은 패턴의 질문
5. 전략을 🆑로 바꾸고 같은 패턴의 질문

### 🤔 판단해보기

세 전략의 체감을 비교한 뒤 *2-3줄로* 답해보세요:

1. **사용자에게 가장 좋은 UX**는 어느 것인가? (🅰️ / 🅱️ / 🆑)
2. 🆑(일괄 표시)의 *전체 완료 시간*은 🅰️보다 *빨랐나요* 느렸나요? *답답함*은?
3. 🅱️(배치)는 🅰️와 🆑 사이에 어떻게 위치하나요? 어떤 상황에서 유용할까요?

> 💡 **이 연습의 진짜 의미**: 스트리밍은 *진짜로 빠른 게 아니라 빨라 보이는* 기술입니다.  
> 같은 5초가 걸려도 *기다림의 질*이 완전히 다르죠. 이게 챗봇 UX의 핵심이고,  
> 7교시 자율 실습에서 본인 챗봇에 *어떤 전략을 쓸지* 직접 선택해야 합니다.

---

## 🏁 7교시 준비 체크리스트

도전과제 2번까지 끝났으면 다음을 확인하세요:

- [ ] 페르소나 4종 드롭다운 동작 (본인이 만든 4번째 페르소나 포함)
- [ ] 응답 스트리밍 (🅰️ 전략으로 글자 단위 표시)
- [ ] 메모리 유지 (이름·관심사 기억)
- [ ] 페르소나별 메모리 분리 (`session_id=f"persona_{{persona}}"`)
- [ ] 3가지 스트리밍 전략의 *체감 차이*를 본인 입으로 설명할 수 있음

### 7교시에 본인이 추가할 것 (Gold 등급)

- 페르소나 5종 이상으로 확장
- 페르소나 설명 자동 표시 (`Dropdown.change` 이벤트 활용)
- 🗑 대화 초기화 버튼 (5교시 도전과제 2 — *둘 다 비우는* 전략)
- 토큰 사용량/비용 표시 (Step 8 코드 활용)
- 본인만의 고유 페르소나 (캐릭터, 역사 인물 등)

수고하셨습니다 🚀